In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 12345

# Load data

In [3]:
breast_cancer = load_breast_cancer(as_frame=True)
breast_cancer_df = breast_cancer.frame
feature_cols = [i.replace(' ', '_') for i in breast_cancer.feature_names]
target_cols = ['target']
breast_cancer_df.columns = feature_cols + target_cols
breast_cancer_df.head()

,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
X = breast_cancer_df[feature_cols].values
y = breast_cancer_df[target_cols].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=RANDOM_STATE)

print('Train', X_train.shape, y_train.shape)
print('Valid', X_valid.shape, y_valid.shape)
print('Test', X_test.shape, y_test.shape)

Train (455, 30) (455,)
Valid (57, 30) (57,)
Test (57, 30) (57,)


# LGBMClassifier

In [5]:
train_data = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
valid_data = lgb.Dataset(X_valid, label=y_valid, feature_name=feature_cols)
test_data = lgb.Dataset(X_test, label=y_test, feature_name=feature_cols)

In [6]:
params = {
    'boosting_type': 'gbdt',       # Gradient Boosting Decision Tree
    'objective': 'binary',         # binary log loss; raw model output is a logit
    'n_estimators': 3,             # Number of boosting rounds
    'num_leaves': 4,               # Max tree leaves for base learners
}

classifier = lgb.train(train_set=train_data, params=params)

[LightGBM] [Info] Number of positive: 285, number of negative: 170
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000320 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4542
[LightGBM] [Info] Number of data points in the train set: 455, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.626374 -> initscore=0.516691
[LightGBM] [Info] Start training from score 0.516691


In [7]:
def print_tree(tree):
    for i in tree:
        if not math.isnan(float(i['split_gain'])):
            message = f"[{i['split_feature']}{i['decision_type']}{i['threshold']}], gain={i['split_gain']}"
        else:
            message = f"leaf={i['value']}"
        tab = '    '*(i['node_depth']-1)
        print(f"{tab}{message}")

In [10]:
# Convert all trees to a single DataFrame
tree_df = classifier.trees_to_dataframe()

In [11]:
# View split details for the first tree
tree = tree_df[tree_df['tree_index'] == 0].to_dict(orient='records')
print_tree(tree)

[worst_perimeter<=105.95000000000002], gain=330.0889892578125
    [worst_concave_points<=0.1337], gain=19.209999084472656
        leaf=0.6730146045357606
        leaf=0.46269177292345715
    [mean_concave_points<=0.04923], gain=31.15049934387207
        leaf=0.5161037963665026
        leaf=0.2629169319936248


In [12]:
# View split details for the second tree
tree = tree_df[tree_df['tree_index'] == 1].to_dict(orient='records')
print_tree(tree)

[worst_area<=874.8500000000001], gain=266.281005859375
    [worst_concave_points<=0.14235000000000003], gain=44.944400787353516
        leaf=0.142441388531469
        leaf=-0.12095912871121704
    [mean_compactness<=0.08929000000000002], gain=2.9914400577545166
        leaf=-0.14576651587037287
        leaf=-0.23063006792156188


In [13]:
# View split details for the third tree
tree = tree_df[tree_df['tree_index'] == 2].to_dict(orient='records')
print_tree(tree)

[worst_perimeter<=105.95000000000002], gain=218.88099670410156
    [worst_concave_points<=0.1337], gain=14.64579963684082
        leaf=0.1405841091340704
        leaf=-0.041868239782313355
    [mean_concave_points<=0.04923], gain=18.930700302124023
        leaf=0.0029175565105158062
        leaf=-0.19334412696827052


# Algorithm

LightGBM is a gradient-boosted decision tree (GBDT). Like XGBoost it builds an additive model in **logit space**,

$$F_t(x) = F_{t-1}(x) + \eta\, f_t(x)$$

where $f_t$ is the $t$-th tree, $\eta$ is the learning rate (shrinkage), and the predicted probability is $p = \sigma(F) = 1/(1+e^{-F})$.

## Loss, gradient and Hessian (binary log loss)

For $y_i\in\{0,1\}$ with logit $\hat y_i = F(x_i)$ and $p_i = \sigma(\hat y_i)$:

$$l(y_i,\hat y_i) = -\big[\,y_i\log p_i + (1-y_i)\log(1-p_i)\,\big]$$

$$g_i = \frac{\partial l}{\partial \hat y_i} = p_i - y_i \qquad h_i = \frac{\partial^2 l}{\partial \hat y_i^2} = p_i(1-p_i)$$

The **pseudo-residual** is the negative gradient, $r_i = y_i - p_i = -g_i$.

## Baseline

The initial score is the optimal constant logit — the log-odds of the positive rate:

$$p_0 = \frac{1}{n}\sum_{i=1}^{n} y_i, \qquad F_0 = \log\frac{p_0}{1-p_0}$$

## Split gain and leaf weight

With aggregated gradient $G=\sum g_i$ and Hessian $H=\sum h_i$ over a node (using $\lambda=$ `reg_lambda`, default $0$), the per-node similarity score and the optimal leaf weight are

$$S = \frac{G^2}{H+\lambda}, \qquad w^{*} = -\frac{G}{H+\lambda} = \frac{\sum_{i}(y_i-p_i)}{\sum_{i} p_i(1-p_i)+\lambda}$$

A candidate split is scored by

$$\text{gain} = S_L + S_R - S_{\text{parent}}$$

(LightGBM's reported `split_gain` uses this form; the constant $\tfrac{1}{2}$ that appears in the loss-reduction derivation is not shown in the reported value, and since it is a constant it does not change which split wins.) Leaf-wise growth then repeatedly splits the leaf with the largest `gain`. The prediction update applies the shrunk leaf weight:

$$F_t(x) = F_{t-1}(x) + \eta\, w^{*}$$

> **Note on the `value` column.** `trees_to_dataframe()` reports each node's `value` as the raw score a sample would have if it stopped at that node. The first tree folds in the baseline, so its leaves show $F_0 + \eta\, w^{*}$ from the second tree on the baseline is $0$, so their leaves show just $\eta\, w^{*}$.

In [14]:
def greedy_find_bin(distinct_values, counts, num_distinct_values, max_bin, total_cnt, min_data_in_bin=3):
    """ Returns the list of bin upper edges (candidate split thresholds). When there are fewer
    distinct values than bins it just places a midpoint every `min_data_in_bin` samples; otherwise
    it greedily merges values into ~equal-frequency bins. `mean_bin_size` is float, 
    so we deliberately use float division to reproduce the exact edges."""
    
    bin_upper_bound = []
    if num_distinct_values <= max_bin:
        cur_cnt_inbin = 0
        for i in range(num_distinct_values - 1):
            cur_cnt_inbin += counts[i]
            if cur_cnt_inbin >= min_data_in_bin:
                val = (distinct_values[i] + distinct_values[i + 1]) / 2.0
                bin_upper_bound.append(val)
                cur_cnt_inbin = 0
    else:
        if min_data_in_bin > 0:
            max_bin = min(max_bin, int(total_cnt / min_data_in_bin))
            max_bin = max(max_bin, 1)
            
        mean_bin_size = total_cnt / max_bin
        rest_bin_cnt = max_bin
        rest_sample_cnt = total_cnt
        is_big_count_value = [False] * num_distinct_values
        
        for i in range(num_distinct_values):
            if counts[i] >= mean_bin_size:
                is_big_count_value[i] = True
                rest_bin_cnt -= 1
                rest_sample_cnt -= counts[i]
                
        mean_bin_size = rest_sample_cnt / rest_bin_cnt
        upper_bounds = [np.inf] * max_bin
        lower_bounds = [np.inf] * max_bin
        bin_cnt = 0
        lower_bounds[bin_cnt] = distinct_values[0]
        cur_cnt_inbin = 0
        
        for i in range(num_distinct_values - 1):
            if not is_big_count_value[i]:
                rest_sample_cnt -= counts[i]
            cur_cnt_inbin += counts[i]
            cond = is_big_count_value[i + 1] & (cur_cnt_inbin >= max(1.0, mean_bin_size * 0.5))
            if is_big_count_value[i] | (cur_cnt_inbin >= mean_bin_size) | cond:
                upper_bounds[bin_cnt] = distinct_values[i]
                bin_cnt += 1
                lower_bounds[bin_cnt] = distinct_values[i + 1]
                if bin_cnt >= max_bin - 1:
                    break
                cur_cnt_inbin = 0
                if not is_big_count_value[i]:
                    rest_bin_cnt -= 1
                    mean_bin_size = rest_sample_cnt / rest_bin_cnt
                    
        bin_cnt += 1
        for i in range(bin_cnt - 1):
            val = float((upper_bounds[i] + lower_bounds[i + 1]) / 2.0)
            if not bin_upper_bound or val > bin_upper_bound[-1]:
                bin_upper_bound.append(val)
                
    return bin_upper_bound

def find_bin(Xf, max_bin=255, min_data_in_bin=3, KZERO=1e-35):
    """Split the value range around zero, reserve one bin as the zero boundary, 
    and distribute the remaining bins between the negative and positive sides 
    proportionally to their sample counts."""

    # sort the column and count how many training samples take each value.
    distinct_values, counts = np.unique(Xf, return_counts=True)
    num_distinct_values = len(distinct_values)
    total_sample_cnt = len(Xf)

    # split the value range around zero into negetive (left), zero (center) and positive (right)
    left_cnt_data = int(np.sum(counts[distinct_values <= -KZERO]))
    cnt_zero = int(np.sum(counts[(distinct_values > -KZERO) & (distinct_values <= KZERO)]))
    right_cnt_data = int(np.sum(counts[distinct_values > KZERO]))

    left_cnt = -1
    for i in range(num_distinct_values):
        if distinct_values[i] > -KZERO:
            left_cnt = i
            break
            
    if left_cnt < 0:
        left_cnt = num_distinct_values

    bounds = []
    if left_cnt > 0 and max_bin > 1:
        left_max_bin = max(1, int(left_cnt_data / (total_sample_cnt - cnt_zero) * (max_bin - 1)))
        left_bounds = greedy_find_bin(distinct_values, counts, left_cnt, left_max_bin, left_cnt_data, min_data_in_bin)
        if len(left_bounds) > 0:
            left_bounds[-1] = -KZERO
        bounds += left_bounds

    right_start = -1
    for i in range(left_cnt, num_distinct_values):
        if distinct_values[i] > KZERO:
            right_start = i
            break

    right_max_bin = max_bin - 1 - len(bounds)
    if right_start >= 0 and right_max_bin > 0:
        right_bounds = greedy_find_bin(distinct_values[right_start:], counts[right_start:],
                                       num_distinct_values - right_start, right_max_bin,
                                       right_cnt_data, min_data_in_bin)
        bounds.append(KZERO)
        bounds += right_bounds
    else:
        bounds.append(np.inf)
    return bounds

In [15]:
# # LightGBM defaults used by the from-scratch reconstruction
# LEARNING_RATE = 0.1                 # shrinkage (eta)
# REG_LAMBDA = 0.0                    # L2 penalty on leaf weights
# MIN_DATA_IN_LEAF = 20               # min samples required in a leaf
# MIN_SUM_HESSIAN_IN_LEAF = 1e-3      # min sum of Hessians (child weight)

# def greedy_find_bin(Xf, max_bin, min_data_in_bin=3):
#     """LightGBM-style histogram binning: returns the list of bin upper edges
#     (candidate split thresholds) for one feature."""
#     total_cnt = len(Xf)
#     distinct_values, counts = np.unique(Xf, return_counts=True)
#     num_distinct_values = len(distinct_values)
#     bin_upper_bound = []
#     if num_distinct_values <= max_bin:
#         cur_cnt_inbin = 0
#         for i in range(num_distinct_values - 1):
#             cur_cnt_inbin += counts[i]
#             if cur_cnt_inbin >= min_data_in_bin:
#                 val = (distinct_values[i] + distinct_values[i + 1]) / 2.0
#                 bin_upper_bound.append(val)
#                 cur_cnt_inbin = 0
#     else:
#         max_bin = min(max_bin, int(total_cnt / min_data_in_bin))
#         max_bin = max(max_bin, 1)
#         mean_bin_size = total_cnt / max_bin
#         rest_bin_cnt = max_bin
#         rest_sample_cnt = total_cnt
#         is_big_count_value = [False] * num_distinct_values
#         for i in range(num_distinct_values):
#             if counts[i] >= mean_bin_size:
#                 is_big_count_value[i] = True
#                 rest_bin_cnt -= 1
#                 rest_sample_cnt -= counts[i]
#         mean_bin_size = rest_sample_cnt / rest_bin_cnt
#         upper_bounds = [np.inf] * max_bin
#         lower_bounds = [np.inf] * max_bin
#         bin_cnt = 0
#         lower_bounds[bin_cnt] = distinct_values[0]
#         cur_cnt_inbin = 0
#         for i in range(num_distinct_values - 1):
#             if not is_big_count_value[i]:
#                 rest_sample_cnt -= counts[i]
#             cur_cnt_inbin += counts[i]
#             cond = is_big_count_value[i + 1] & (cur_cnt_inbin >= max(1.0, mean_bin_size * 0.5))
#             if is_big_count_value[i] | (cur_cnt_inbin >= mean_bin_size) | cond:
#                 upper_bounds[bin_cnt] = distinct_values[i]
#                 bin_cnt += 1
#                 lower_bounds[bin_cnt] = distinct_values[i + 1]
#                 if bin_cnt >= max_bin - 1:
#                     break
#                 cur_cnt_inbin = 0
#                 if not is_big_count_value[i]:
#                     rest_bin_cnt -= 1
#                     mean_bin_size = rest_sample_cnt / rest_bin_cnt
#         bin_cnt += 1
#         for i in range(bin_cnt - 1):
#             bin_upper_bound.append(float((upper_bounds[i] + lower_bounds[i + 1]) / 2.0))
#     return bin_upper_bound

# # Build the candidate thresholds (bin edges) once, per feature, from the training data.
# possible_thresholds = dict()
# for idx in range(len(feature_cols)):
#     possible_thresholds[idx] = greedy_find_bin(X_train[:, idx], max_bin=255)

In [16]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def similarity_score(residuals, hessians, reg_lambda=0.0):
    """S = G^2 / (H + lambda).  residual = y - p = -g, so G^2 is the same either sign."""
    G = np.sum(residuals)
    H = np.sum(hessians)
    return (G * G) / (H + reg_lambda)

def leaf_value(residuals, hessians, reg_lambda=0.0, learning_rate=0.1):
    """Shrunk optimal leaf weight:  eta * w* = eta * sum(y-p) / (sum(p(1-p)) + lambda)."""
    G = np.sum(residuals)
    H = np.sum(hessians)
    return learning_rate * G / (H + reg_lambda)

In [17]:
def find_best_split(X, residuals, hessians, feature_cols, possible_thresholds,
                    reg_lambda=0.0, min_data_in_leaf=20,
                    min_sum_hessian_in_leaf=1e-3):
    similarity_parent = similarity_score(residuals, hessians, reg_lambda)

    max_gain = -1
    best = None
    for idx in range(len(feature_cols)):
        Xf = X[:, idx]
        for thres in possible_thresholds[idx]:
            # split the node into a "left" (<=) group and a "right" (>) group
            mask_left = Xf <= thres
            mask_right = ~mask_left
            n_left = int(mask_left.sum())
            n_right = int(mask_right.sum())
            if n_left == 0 or n_right == 0:
                continue
            if n_left < min_data_in_leaf or n_right < min_data_in_leaf:
                continue

            r_left, h_left = residuals[mask_left], hessians[mask_left]
            r_right, h_right = residuals[mask_right], hessians[mask_right]
            H_left = np.sum(h_left)
            H_right = np.sum(h_right)
            if H_left < min_sum_hessian_in_leaf or H_right < min_sum_hessian_in_leaf:
                continue

            sim_left = similarity_score(r_left, h_left, reg_lambda)
            sim_right = similarity_score(r_right, h_right, reg_lambda)
            gain = sim_left + sim_right - similarity_parent

            if (gain > max_gain) and (gain > 0):
                max_gain = gain
                best = {
                    'feature': feature_cols[idx], 'feature_idx': idx,
                    'threshold': thres, 'gain': gain,
                    'left':  (X[mask_left],  r_left,  h_left),
                    'right': (X[mask_right], r_right, h_right),
                }
    return best

def find_next_split_node(leaf_nodes, feature_cols, possible_thresholds):
    """Leaf-wise (best-first) growth: pick the existing leaf with the largest split gain."""
    best_idx = None
    best_split = None
    best_gain = -1
    for idx, node in enumerate(leaf_nodes):
        Xn, rn, hn = node[1], node[2], node[3]
        split = find_best_split(Xn, rn, hn, feature_cols, possible_thresholds)
        if split is not None and split['gain'] > best_gain:
            best_gain = split['gain']
            best_idx = idx
            best_split = split
    return best_idx, best_split

In [18]:
def build_tree(X, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=0.0):
    """Grow a tree leaf-wise (best-first) until num_leaves is reached.

    Returns the root node. Each node is a dict with feature/threshold/gain/
    left/right/depth; leaves additionally carry a `value`."""
    root = {'feature': None, 'threshold': None, 'gain': None,
            'left': None, 'right': None, 'depth': 0}
    # each leaf entry is (node_dict, X_subset, residuals_subset, hessians_subset)
    leaves = [(root, X, residuals, hessians)]
    while len(leaves) < num_leaves:
        best_idx, best_split = find_next_split_node(leaves, feature_cols, possible_thresholds)
        if best_split is None:
            break
        node, _, _, _ = leaves.pop(best_idx)
        node['feature'] = best_split['feature']
        node['threshold'] = best_split['threshold']
        node['gain'] = best_split['gain']
        d = node['depth']
        left_child = {'feature': None, 'threshold': None, 'gain': None, 'left': None, 'right': None, 'depth': d + 1}
        right_child = {'feature': None, 'threshold': None, 'gain': None, 'left': None, 'right': None, 'depth': d + 1}
        node['left'] = left_child
        node['right'] = right_child
        leaves.insert(best_idx, (right_child, best_split['right'][0], best_split['right'][1], best_split['right'][2]))
        leaves.insert(best_idx, (left_child, best_split['left'][0], best_split['left'][1], best_split['left'][2]))
        
    for (node, _, rn, hn) in leaves:
        node['value'] = init_score + leaf_value(rn, hn)
        
    return root

def print_scratch_tree(node, indent=0):
    pad = '    ' * indent
    if node['left'] is None:
        print(f"{pad}leaf={node['value']}")
    else:
        print(f"{pad}[{node['feature']}<={node['threshold']:.8f}], gain={node['gain']}")
        print_scratch_tree(node['left'], indent + 1)
        print_scratch_tree(node['right'], indent + 1)

## Build candidate thresholds of each feature.

LightGBM never scans every distinct value when searching for a split. Instead, *once up front*,
it reduces each feature to a small set of **histogram bins**, and the split search later only ever tests the **bin edges** as candidate thresholds. 

`find_bin()` turns one feature column `Xf` into that list of candidate thresholds, which we store in `possible_thresholds`.

1. sort the column and count how many training samples take each value.
2. split the value range around zero into negetive (left), zero (center) and positive (right)
3. compute the number of bins for negative and positive side. They are divided proportionally to sample counts
4. bin the negative side and force the last edge to -1e-35 so the negative region ends exactly at the zero boundary.
5. bin the zero side at 1e-35
6. bin the positve side

`greedy_find_bin()` used to find the candidate thresholds for both negative and positive side.
- **Few distinct values (`num_distinct_values <= max_bin`):**  
    walk the sorted values and drop a
    midpoint `(v[i] + v[i+1])/2` every time a running count reaches `min_data_in_bin` (default=3).
  
- **Many distinct values:**  
    greedily merge consecutive values into roughly **equal-frequency** bins. The target fill is `mean_bin_size = total_cnt / max_bin`.  
    Any single value whose own count already exceeds it (`is_big_count_value`) gets its own bin.  
    A new edge is opened once the running count reaches `mean_bin_size`.  
    The edge value is the midpoint of the two boundary distinct values, `(upper_bounds[i] + lower_bounds[i+1]) / 2`.

In [22]:
# Build the candidate thresholds (bin edges) once, per feature, from the training data.
possible_thresholds = dict()
for idx in range(len(feature_cols)):
    possible_thresholds[idx] = find_bin(X_train[:, idx], max_bin=255)

## Tree 0 — initialize the baseline

Start from the constant log-odds $F_0 = \log\frac{p_0}{1-p_0}$ with $p_0 = \bar y$, convert it to a probability, and form the first pseudo-residuals $r_i = y_i - p_0$ and Hessians $h_i = p_0(1-p_0)$.

In [23]:
p0 = y_train.mean()
logit0 = np.log(p0 / (1 - p0))
F_prev = np.full(len(y_train), logit0)   # F0: constant initial log-odds

p = sigmoid(F_prev)
residuals = y_train - p
hessians = p * (1 - p)

## Fit tree 0

Grow leaf-wise (always split the leaf with the largest gain). The first tree folds the baseline into its leaf `value`s, so each leaf shows $F_0 + \eta\, w^{*}$ — matching LightGBM's `trees_to_dataframe()` output for `tree_index == 0` above.

In [24]:
root = build_tree(X_train, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=logit0)
print_scratch_tree(root)

[worst_perimeter<=105.95000000], gain=330.0887063734625
    [worst_concave_points<=0.13370000], gain=19.20999121352304
        leaf=0.67301460391155
        leaf=0.4626917752101329
    [mean_concave_points<=0.04923000], gain=31.150518275424815
        leaf=0.5161037979139512
        leaf=0.26291693704520186


## Update predictions

Add the first tree's contribution to the logits. `predict(..., raw_score=True)` returns **logits** (the probabilities are just `sigmoid` of these), so we use it directly as $F_1$ and recompute probabilities and residuals for the next tree.

In [25]:
F_prev = classifier.predict(X_train, num_iteration=1, raw_score=True)   # F1 (logits)

## Tree 1

Recompute $p = \sigma(F_1)$, residuals $y - p$, and Hessians $p(1-p)$, then grow the second tree. From the second tree on the baseline is $0$, so leaves show just $\eta\, w^{*}$.

In [26]:
p = sigmoid(F_prev)
residuals = y_train - p
hessians = p * (1 - p)

root = build_tree(X_train, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=0.0)
print_scratch_tree(root)

[worst_area<=874.85000000], gain=266.28052141709026
    [worst_concave_points<=0.14235000], gain=44.94437896452871
        leaf=0.14244139478583004
        leaf=-0.12095912488465736
    [mean_compactness<=0.08929000], gain=2.9914386191829294
        leaf=-0.14576651312996533
        leaf=-0.2306300592040255


In [27]:
F_prev = classifier.predict(X_train, num_iteration=2, raw_score=True)   # F2 (logits)

## Tree 2

One more boosting round, using $F_2$ as the current logits.

In [28]:
p = sigmoid(F_prev)
residuals = y_train - p
hessians = p * (1 - p)

root = build_tree(X_train, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=0.0)
print_scratch_tree(root)

[worst_perimeter<=105.95000000], gain=218.8805123749607
    [worst_concave_points<=0.13370000], gain=14.6458197533764
        leaf=0.14058410845926675
        leaf=-0.0418682347301381
    [mean_concave_points<=0.04923000], gain=18.930705361807995
        leaf=0.0029175581677598913
        leaf=-0.19334412626954248
